# Stage 1 — Soft-normalize annotator corrections

Map each completed top-1 correction onto the 16-class MIDAS `y16` label.

1. Rule-based normalization (abbreviations, synonyms → canonical y16)
2. Join MIDAS pathology ground truth by `case_id`
3. Where rule ≠ pathology, optionally ask GPT-4o if the free text is a reasonable match
4. Write `data/normed/edits_soft_normalized.csv`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "editing" / "shared"))
sys.path.insert(0, str(Path(".").resolve()))

import config as C
from data_engineering.normalize_dx import normalize
from data_engineering.parse_annotations import filter_records, load_records
from gt import load as load_gt

In [ ]:
OUT_DIR = REPO_ROOT / "data" / "normed"
OUT_CSV = OUT_DIR / "edits_soft_normalized.csv"
LLM_CACHE = OUT_DIR / "llm_judgments.json"

USE_LLM = True   # False = rule-based labels only, no API calls

## 1. Build rows — rule-based normalization + ground truth

In [ ]:
import pandas as pd

gt = load_gt()
records = filter_records(
    load_records(C.ANNOTATIONS_CSV),
    require_complete=True,
    top1_errors_only=True,
    require_correction=True,
)

rows = []
for rec in records:
    nd = normalize(rec.human_top1_raw)
    rows.append({
        "login_id": rec.login_id,
        "case_id": rec.case_id,
        "model": rec.model,
        "model_top1": rec.model_top1,
        "human_top1_raw": rec.human_top1_raw,
        "human_top1_is_multi": nd.is_multi,
        "human_top1_norm": nd.norm,
        "human_top1_y16_rule": nd.y16,
        "gt_y16": gt.get(rec.case_id),
        "llm_match": None,
        "llm_reason": None,
        "human_top1_y16_soft": None,
    })

df = pd.DataFrame(rows)
print(f"{len(df)} corrections loaded")
df.head()

## 2. Summary — rule match vs mismatch

In [ ]:
n_rule_match = (df["human_top1_y16_rule"] == df["gt_y16"]).sum()
n_no_gt = df["gt_y16"].isna().sum()
n_mismatch = len(df) - n_rule_match - n_no_gt

print(f"Rule matches GT:  {n_rule_match}")
print(f"Mismatch (→ LLM): {n_mismatch}")
print(f"No GT available:  {n_no_gt}")

### Mismatch rows (these go to GPT-4o if `USE_LLM=True`)

In [ ]:
mismatches = df[
    df["gt_y16"].notna() & (df["human_top1_y16_rule"] != df["gt_y16"])
][["case_id", "human_top1_raw", "human_top1_y16_rule", "gt_y16"]]
print(f"{len(mismatches)} mismatches")
mismatches

## 3. Apply soft normalization

In [ ]:
import json

cache = json.loads(LLM_CACHE.read_text()) if LLM_CACHE.exists() else {}
n_api = 0

for row in rows:
    rule = row["human_top1_y16_rule"]
    gt_val = row["gt_y16"]

    if gt_val is None:
        row["human_top1_y16_soft"] = rule
        continue

    if rule == gt_val:
        row["llm_match"] = True
        row["human_top1_y16_soft"] = gt_val
        continue

    cache_key = f"{row['human_top1_raw']}||{gt_val}"

    if cache_key in cache:
        verdict = cache[cache_key]
    elif not USE_LLM:
        row["human_top1_y16_soft"] = rule
        continue
    else:
        from llm_judge import judge
        verdict = judge(row["human_top1_raw"], gt_val)
        cache[cache_key] = verdict
        n_api += 1

    row["llm_match"] = verdict.get("match", False)
    row["llm_reason"] = verdict.get("reason", "")
    row["human_top1_y16_soft"] = gt_val if verdict.get("match") else rule

if n_api:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    LLM_CACHE.write_text(json.dumps(cache, indent=2, ensure_ascii=False))
    print(f"{n_api} new LLM API calls (cached for next run)")

df = pd.DataFrame(rows)

n_rule_match = (df["human_top1_y16_rule"] == df["gt_y16"]).sum()
n_llm_match = ((df["llm_match"] == True) & (df["human_top1_y16_rule"] != df["gt_y16"])).sum()
n_no_gt = df["gt_y16"].isna().sum()
n_wrong = len(df) - n_rule_match - n_llm_match - n_no_gt

print(f"Rule match:    {n_rule_match}")
print(f"LLM promoted:  {n_llm_match}")
print(f"Genuine error: {n_wrong}")
print(f"No GT:         {n_no_gt}")

### Inspect LLM-promoted rows

In [ ]:
promoted = df[
    (df["llm_match"] == True) & (df["human_top1_y16_rule"] != df["gt_y16"])
][["case_id", "human_top1_raw", "human_top1_y16_rule", "gt_y16", "llm_reason"]]
print(f"{len(promoted)} rows promoted by LLM")
promoted

## 4. Write CSV

In [ ]:
import csv

COLUMNS = [
    "login_id", "case_id", "model",
    "model_top1", "human_top1_raw", "human_top1_is_multi",
    "human_top1_norm", "human_top1_y16_rule", "gt_y16",
    "llm_match", "llm_reason",
    "human_top1_y16_soft",
]

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

print(f"Wrote {len(rows)} rows → {OUT_CSV}")